# Neural Network from Scratch (NumPy Only)

Build a 2-layer neural network for binary classification using only NumPy.
This will give you deep intuition for how forward pass and backpropagation work.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# Generate dataset
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)

In [ ]:
class NeuralNetworkFromScratch:
    def __init__(self, input_size, hidden_size, output_size, lr=0.01):
        self.lr = lr
        # He initialization
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2 / input_size)
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2 / hidden_size)
        self.b2 = np.zeros((1, output_size))
        self.losses = []

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def relu(self, z):
        return np.maximum(0, z)

    def relu_derivative(self, z):
        return (z > 0).astype(float)

    def forward(self, X):
        self.Z1 = X @ self.W1 + self.b1
        self.A1 = self.relu(self.Z1)
        self.Z2 = self.A1 @ self.W2 + self.b2
        self.A2 = self.sigmoid(self.Z2)
        return self.A2

    def compute_loss(self, y_true, y_pred):
        m = y_true.shape[0]
        y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
        loss = -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
        return loss

    def backward(self, X, y):
        m = X.shape[0]
        y = y.reshape(-1, 1)

        # Output layer gradients
        dZ2 = self.A2 - y
        dW2 = (self.A1.T @ dZ2) / m
        db2 = np.mean(dZ2, axis=0, keepdims=True)

        # Hidden layer gradients
        dA1 = dZ2 @ self.W2.T
        dZ1 = dA1 * self.relu_derivative(self.Z1)
        dW1 = (X.T @ dZ1) / m
        db1 = np.mean(dZ1, axis=0, keepdims=True)

        # Update weights
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    def train(self, X, y, n_epochs=1000):
        for epoch in range(n_epochs):
            y_pred = self.forward(X)
            loss = self.compute_loss(y, y_pred)
            self.losses.append(loss)
            self.backward(X, y)
            if epoch % 200 == 0:
                print(f'Epoch {epoch}, Loss: {loss:.4f}')

    def predict(self, X, threshold=0.5):
        return (self.forward(X) >= threshold).astype(int).flatten()

# Train
model = NeuralNetworkFromScratch(input_size=2, hidden_size=16, output_size=1, lr=0.1)
model.train(X_train, y_train, n_epochs=1000)

train_acc = (model.predict(X_train) == y_train).mean()
test_acc = (model.predict(X_test) == y_test).mean()
print(f'\nTrain Accuracy: {train_acc:.4f}')
print(f'Test Accuracy: {test_acc:.4f}')

plt.plot(model.losses)
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Binary Cross-Entropy')
plt.grid(True)
plt.show()